#  Resume ↔ Job Description Matching

This notebook implements the Resume-to-Job Description Matching component of **SkillSync AI**.

The important correction in this version is that the matching pipeline preserves legitimate professional/domain skills such as **communication, customer service, recruitment, performance management, benefits administration, and administration** instead of allowing generic-word filtering to remove them.

The system remains domain-independent: the same matching function can be used for HR, IT, finance, healthcare, engineering, marketing, and other fields.


In [21]:
import sys
import pandas as pd

sys.path.append("../src")

from skill_extraction import (
    build_extraction_vocabulary,
    extract_and_classify_skills,
    match_resume_to_jd,
    supplemental_professional_skills,
)

print("Corrected SkillSync AI matching module loaded successfully.")


Corrected SkillSync AI matching module loaded successfully.


## Load the cleaned skill vocabulary

`skills_clean.csv` remains the cleaned source vocabulary. We do not overwrite it.

The extraction vocabulary is built at runtime so that source data cleaning and application-specific extraction rules remain separate.


In [22]:
skills_df = pd.read_csv("../data/processed/skills_clean.csv")

generic_words = {
    "advanced", "analysis", "analytical", "codes", "collateral",
    "conflict", "customer", "data", "design", "features", "general",
    "government", "health", "hospital", "hotel", "insurance",
    "interpret", "layout", "management", "office", "operations",
    "paperwork", "participation", "plan", "policies", "prevention",
    "print", "public", "ratings", "records", "reporting", "reviews",
    "safety", "sales", "service", "skills", "statistics", "system",
    "teams", "terminology", "training", "transactions", "website",
    "organization", "professional", "responsibilities", "experience",
    "work", "working", "software"
}

protected_skills = {
    "python", "java", "javascript", "c", "c++", "c#", "f#", "g++",
    "sql", "html", "css", "php", "ruby", "swift", "kotlin", "go", "r",
    "matlab", "scala", "perl", "bash", "powershell", "git", "github",
    "gitlab", "npm", "node.js", "react.js", "vue", "angular", "mongodb",
    "mysql", "postgresql", "oracle", "redis", "numpy", "pandas",
    "tensorflow", "pytorch", "keras", "tableau", "power bi", "docker",
    "kubernetes", "aws", "azure", "gcp", "linux", "android", "ios",
    "sap", "excel", "jira", "front-end"
}

skill_set = build_extraction_vocabulary(
    skills_df,
    generic_words=generic_words,
    protected_skills=protected_skills,
)

print("Source vocabulary:", len(skills_df))
print("Final extraction vocabulary:", len(skill_set))
print("Supplemental professional skills:", len(supplemental_professional_skills))


Source vocabulary: 36167
Final extraction vocabulary: 36106
Supplemental professional skills: 19


## Load candidate resume

For validation, the first HR resume from the cleaned resume dataset is used. In the final application, this text will come from the uploaded resume.


In [23]:
resume_df = pd.read_csv("../data/processed/resumes_clean.csv")

candidate_resume = resume_df.iloc[0]["clean_resume"]

print("Total resumes:", len(resume_df))
print("Candidate category:", resume_df.iloc[0]["Category"])


Total resumes: 2484
Candidate category: HR


## HR Job Description validation

This is a validation case only. The production application will accept any Job Description supplied by the user.


In [24]:
hr_job_description = """
We are looking for an HR Administrator with experience in
human resources, employee relations, recruitment, compensation,
labor relations, benefits administration, payroll, performance
management, documentation, employee records, conflict resolution,
training and development, customer service, and communication.

The candidate should have strong organizational skills,
time management, leadership and administrative abilities.
Experience with healthcare benefits, pension plans and
employee assistance programs is preferred.
"""

generic_jd_skills = {
    "building",
    "models",
    "strong organizational skills",
}


## Run the corrected matching workflow

The matcher calculates:

- total resume skills
- total JD skills
- matched skills
- missing skills
- additional resume skills
- skill match percentage

The percentage is calculated as:

**matched JD skills / total validated JD skills × 100**

This measures skill overlap only; it is not a hiring decision.


In [25]:
hr_result = match_resume_to_jd(
    candidate_resume,
    hr_job_description,
    skill_set,
    generic_jd_skills,
)

print("HR Resume ↔ HR Job Description")
print("=" * 45)

print("Total Resume Skills:", hr_result["total_resume_skills"])
print("Total JD Skills:", hr_result["total_jd_skills"])
print("Matched Skills:", hr_result["matched_count"])
print("Missing Skills:", hr_result["missing_count"])
print("Additional Skills:", hr_result["additional_count"])
print("Skill Match Percentage:",
      f'{hr_result["skill_match_percentage"]}%')

print("\nMatched Skills:")
for skill in hr_result["matched_skills"]:
    print("-", skill)

print("\nMissing Skills:")
for skill in hr_result["missing_skills"]:
    print("-", skill)


HR Resume ↔ HR Job Description
Total Resume Skills: 57
Total JD Skills: 17
Matched Skills: 12
Missing Skills: 5
Additional Skills: 45
Skill Match Percentage: 70.59%

Matched Skills:
- compensation
- conflict resolution
- documentation
- employee relations
- healthcare
- human resources
- labor relations
- leadership
- payroll
- pension
- time management
- training and development

Missing Skills:
- benefits administration
- communication
- customer service
- performance management
- recruitment


## General-field validation

The same `match_resume_to_jd()` function should be reused for other domains.

Examples for later validation:

- Information Technology
- Finance
- Healthcare
- Engineering
- Marketing
- Education

Only the resume and Job Description change. The matching logic does not need to be rewritten for each field.


In [26]:
# Example of the reusable interface:

# result = match_resume_to_jd(
#     resume_text,
#     job_description,
#     skill_set,
#     generic_jd_skills=set()
# )

# print(result["matched_skills"])
# print(result["missing_skills"])
# print(result["skill_match_percentage"])


## Step 16 — Multi-Field Validation

SkillSync AI is designed as a domain-independent resume screening system.

The matching function should work with Job Descriptions from different professional fields without changing the underlying matching logic.

To validate this, the same `match_resume_to_jd()` function is tested with multiple professional domains.

The purpose of this test is to verify that:

- The system can extract skills from different types of Job Descriptions.
- The system can extract skills from resumes belonging to different fields.
- Matched skills are correctly identified.
- Missing skills are correctly identified.
- Additional resume skills are correctly identified.
- The skill match percentage is calculated consistently.

The Job Description and resume are the only inputs that change between tests.

In [27]:
# Step 16 — Multi-Field Validation

test_cases = {
    "HR": {
        "resume": candidate_resume,
        "job_description": hr_job_description
    }
}

for field, data in test_cases.items():

    result = match_resume_to_jd(
        data["resume"],
        data["job_description"],
        skill_set,
        generic_jd_skills
    )

    print("\n" + "=" * 60)
    print(f"{field} VALIDATION")
    print("=" * 60)

    print("Total Resume Skills:",
          result["total_resume_skills"])

    print("Total JD Skills:",
          result["total_jd_skills"])

    print("Matched Skills:",
          result["matched_count"])

    print("Missing Skills:",
          result["missing_count"])

    print("Additional Skills:",
          result["additional_count"])

    print("Skill Match Percentage:",
          f'{result["skill_match_percentage"]}%')

    print("\nMatched Skills:")

    for skill in result["matched_skills"]:
        print("-", skill)

    print("\nMissing Skills:")

    for skill in result["missing_skills"]:
        print("-", skill)


HR VALIDATION
Total Resume Skills: 57
Total JD Skills: 17
Matched Skills: 12
Missing Skills: 5
Additional Skills: 45
Skill Match Percentage: 70.59%

Matched Skills:
- compensation
- conflict resolution
- documentation
- employee relations
- healthcare
- human resources
- labor relations
- leadership
- payroll
- pension
- time management
- training and development

Missing Skills:
- benefits administration
- communication
- customer service
- performance management
- recruitment


### Finance Validation

The matching system is now tested using a Finance Job Description.

No changes are made to the matching algorithm. Only the resume and Job Description are changed.

This test verifies that the system can identify domain-specific skills and calculate the skill overlap for a field different from Human Resources.

In [28]:
# Finance Job Description

finance_job_description = """
We are looking for a Finance Analyst with experience in
financial analysis, accounting, budgeting, forecasting,
financial reporting, financial management, Excel, data analysis,
and financial modeling.

The candidate should have knowledge of accounts payable,
accounts receivable, general ledger, cost analysis,
and variance analysis.

Experience with Microsoft Excel, financial statements,
and management reporting is required.
"""

In [29]:
# Select a Finance resume

finance_resume_row = resume_df[
    resume_df["Category"] == "FINANCE"
].iloc[0]

finance_resume = finance_resume_row["clean_resume"]

print("Resume ID:", finance_resume_row["ID"])
print("Category:", finance_resume_row["Category"])

Resume ID: 11877150
Category: FINANCE


In [30]:
# Finance validation

finance_result = match_resume_to_jd(
    finance_resume,
    finance_job_description,
    skill_set,
    generic_jd_skills
)

print("\n" + "=" * 60)
print("FINANCE VALIDATION")
print("=" * 60)

print("Total Resume Skills:",
      finance_result["total_resume_skills"])

print("Total JD Skills:",
      finance_result["total_jd_skills"])

print("Matched Skills:",
      finance_result["matched_count"])

print("Missing Skills:",
      finance_result["missing_count"])

print("Additional Skills:",
      finance_result["additional_count"])

print("Skill Match Percentage:",
      f'{finance_result["skill_match_percentage"]}%')

print("\nMatched Skills:")

for skill in finance_result["matched_skills"]:
    print("-", skill)

print("\nMissing Skills:")

for skill in finance_result["missing_skills"]:
    print("-", skill)


FINANCE VALIDATION
Total Resume Skills: 38
Total JD Skills: 15
Matched Skills: 3
Missing Skills: 12
Additional Skills: 35
Skill Match Percentage: 20.0%

Matched Skills:
- accounting
- budgeting
- finance

Missing Skills:
- accounts payable
- accounts receivable
- data analysis
- financial analysis
- financial management
- financial modeling
- financial reporting
- financial statements
- forecasting
- general ledger
- microsoft excel
- variance analysis


In [31]:
# Inspect extracted Finance JD skills

finance_jd_skills = extract_and_classify_skills(
    finance_job_description,
    skill_set
)

print("Total extracted Finance JD skills:",
      len(finance_jd_skills))

print("\nExtracted Finance JD skills:")

for skill in sorted(finance_jd_skills):
    print("-", skill)

Total extracted Finance JD skills: 15

Extracted Finance JD skills:
- accounting
- accounts payable
- accounts receivable
- budgeting
- data analysis
- finance
- financial analysis
- financial management
- financial modeling
- financial reporting
- financial statements
- forecasting
- general ledger
- microsoft excel
- variance analysis


In [32]:
# Inspect extracted Finance resume skills

finance_resume_skills = extract_and_classify_skills(
    finance_resume,
    skill_set
)

print("Total extracted Finance resume skills:",
      len(finance_resume_skills))

print("\nExtracted Finance resume skills:")

for skill in sorted(finance_resume_skills):
    print("-", skill)

Total extracted Finance resume skills: 38

Extracted Finance resume skills:
- access
- accounting
- audit
- birth
- boxing
- budgeting
- c
- communication skills
- controls
- cost savings
- critical thinking
- english
- excel
- expenses
- finance
- financial reports
- financial research
- flow
- height
- history
- interpersonal skills
- jogging
- marketing
- modeling
- multilingual
- other
- petty cash
- process
- project
- projects
- property consultancy
- religion
- sales promotion
- status
- time management
- variances
- word
- yoga


In [33]:
# Inspect the relevant Finance resume text

finance_terms_to_check = [
    "financial reports",
    "financial research",
    "modeling",
    "variances",
    "excel",
    "accounting",
    "budgeting"
]

print("Finance resume term validation")
print("=" * 50)

for term in finance_terms_to_check:
    if term.lower() in finance_resume.lower():
        print(f"✓ {term}")
    else:
        print(f"✗ {term}")

Finance resume term validation
✓ financial reports
✓ financial research
✓ modeling
✓ variances
✓ excel
✓ accounting
✓ budgeting


In [34]:
# Step 16.2 — Test safe skill variations

variation_test = {
    "financial reports": "financial reporting",
    "microsoft excel": "excel",
    "account payable": "accounts payable",
    "account receivable": "accounts receivable",
    "general ledger accounting": "general ledger"
}

print("Skill variation tests")
print("=" * 60)

for resume_term, jd_term in variation_test.items():

    resume_has = resume_term in finance_resume_skills
    jd_has = jd_term in finance_jd_skills

    print(f"{resume_term}  →  {jd_term}")
    print(f"Resume: {resume_has} | JD: {jd_has}")
    print()

Skill variation tests
financial reports  →  financial reporting
Resume: True | JD: True

microsoft excel  →  excel
Resume: False | JD: False

account payable  →  accounts payable
Resume: False | JD: True

account receivable  →  accounts receivable
Resume: False | JD: True

general ledger accounting  →  general ledger
Resume: False | JD: True



In [35]:
# Healthcare Job Description

healthcare_job_description = """
We are looking for a Healthcare Administrator with experience in
healthcare administration, patient records, medical terminology,
customer service, documentation, scheduling, communication,
healthcare management, insurance processing, billing, and
electronic health records.

The candidate should have strong organizational skills,
time management, interpersonal skills, and data management abilities.
Experience with hospital administration and patient coordination
is preferred.
"""

In [36]:
# Select a Healthcare resume

healthcare_resume_row = resume_df[
    resume_df["Category"] == "HEALTHCARE"
].iloc[0]

healthcare_resume = healthcare_resume_row["clean_resume"]

print("Resume ID:", healthcare_resume_row["ID"])
print("Category:", healthcare_resume_row["Category"])

Resume ID: 23617240
Category: HEALTHCARE


In [37]:
# Healthcare validation

healthcare_result = match_resume_to_jd(
    healthcare_resume,
    healthcare_job_description,
    skill_set,
    generic_jd_skills
)

print("\n" + "=" * 60)
print("HEALTHCARE VALIDATION")
print("=" * 60)

print("Total Resume Skills:",
      healthcare_result["total_resume_skills"])

print("Total JD Skills:",
      healthcare_result["total_jd_skills"])

print("Matched Skills:",
      healthcare_result["matched_count"])

print("Missing Skills:",
      healthcare_result["missing_count"])

print("Additional Skills:",
      healthcare_result["additional_count"])

print("Skill Match Percentage:",
      f'{healthcare_result["skill_match_percentage"]}%')

print("\nMatched Skills:")

for skill in healthcare_result["matched_skills"]:
    print("-", skill)

print("\nMissing Skills:")

for skill in healthcare_result["missing_skills"]:
    print("-", skill)


HEALTHCARE VALIDATION
Total Resume Skills: 112
Total JD Skills: 12
Matched Skills: 1
Missing Skills: 11
Additional Skills: 111
Skill Match Percentage: 8.33%

Matched Skills:
- billing

Missing Skills:
- administration
- communication
- customer service
- data management
- documentation
- healthcare management
- interpersonal skills
- medical terminology
- processing
- scheduling
- time management


## Step 17 — Multi-Field Validation Summary

The Resume-to-Job Description matching system was validated using Job Descriptions from multiple professional domains.

The validation included:

- Human Resources
- Finance
- Healthcare

The same `match_resume_to_jd()` function was used for all three domains without modifying the matching algorithm.

The results demonstrate that the system is not restricted to a single professional field. The matching process depends on the skills extracted from the candidate's resume and the skills extracted from the user-provided Job Description.

The system calculates:

- Total resume skills
- Total Job Description skills
- Matched skills
- Missing skills
- Additional resume skills
- Skill match percentage

The validation also identified an important limitation of the current implementation: the matching stage primarily uses exact normalized skill matching. Linguistically similar expressions, such as different grammatical forms or related terminology, may not always be recognized as the same skill.

This limitation will be considered in later NLP improvements.

### Phase 6 Status

- [x] Create sample Job Description
- [x] Extract skills from Job Description
- [x] Extract skills from Resume
- [x] Compare Resume Skills with JD Skills
- [x] Identify matched skills
- [x] Identify missing skills
- [x] Calculate skill match percentage
- [x] Test positive matching case
- [x] Validate HR domain
- [x] Validate Finance domain
- [x] Validate Healthcare domain

**Phase 6 — Resume ↔ Job Description Matching: COMPLETE**